# Companion — Setup & Test

Run top to bottom. Each stage confirms one layer before the next stage depends on it — failures are localised instantly.

Select the **companion_env** kernel in the top-right, or activate it in a terminal:
```bash
source companion_env/bin/activate
```

**Stages**

| Stage | What it proves |
| --- | --- |
| 0 | Dependencies, models, and ESP32 firmware are installed |
| 1 | Readiness probe — every model path and device node present |
| 2 | Unit + integration tests — the new integration layer is wired correctly |
| 3 | Debug GUI — each subsystem (audio / STT / LLM / vision / motor / face) works in isolation |
| 4 | Combined CLI sanity check |
| 5 | Sim-motor dry run of the full app — safe, no motor movement |
| 6 | Real-motor run + touchscreen tiles + telemetry latency check |


## Stage 0 — Install

**Run `scripts/setup.sh` in a terminal first** (it needs sudo — Jupyter can't prompt). It installs system packages, CUDA deps, the CH341 serial driver for the ESP32 screen, builds llama.cpp with CUDA, and sets up the venv.


In [ ]:
import sys, os
os.environ['PATH'] = os.path.dirname(sys.executable) + ':' + os.environ.get('PATH', '')
print('Python:', sys.executable)

# If you haven't run `bash scripts/setup.sh` in a terminal yet, stop now and do that.
# (It needs sudo access for system packages + the CH341 kernel driver.)
!pip install -r requirements.txt


### Download every model

LLM (Gemma 4 E2B + E4B), tool router (FunctionGemma 270M), STT (Parakeet), TTS (Kokoro + Piper), vision (YOLO26n-pose + HSEmotion), EOU, speaker ID (TitaNet-L).

The VLM is now the **multimodal Gemma** itself (via `llm.mmproj_path`), so Moondream is no longer required. The standalone Moondream files stay supported by the debug GUI's VLM tab but are skipped in production.


In [ ]:
!python3 scripts/download_models.py


### Flash the ESP32 face firmware (once)

With the Diymore 2.8" screen plugged into a USB-C port on the Jetson.

**Note:** firmware was edited this round — the "Privacy" and "Status" tiles (which had no host handlers) were removed from the More list. Re-flash if you want the UI to match what the host actually implements.


In [ ]:
!bash scripts/flash_firmware.sh

## Stage 1 — Readiness probe

`scripts/preflight.py` checks every model path in `config.yaml`, every required device node (motor serial, ESP32 serial), and every log directory. Fail-loud before `main.py` gets a chance to boot a half-dead robot.

Expected: every row `[OK]`, ending `READY`.

If anything fails, fix it before moving on. Typical causes:
- missing model file → re-run Stage 0 download
- missing serial device → USB cable not plugged in, or udev rule not installed (see `deploy/udev-rules/`)
- missing `titanet-l.onnx` → download it, or set `speaker_id.enabled: false` in `config.yaml`


In [ ]:
!python3 scripts/preflight.py


## Stage 2 — Unit + integration tests

These run in under 2 seconds and verify the new integration layer — event bus, state machine, Turn lifecycle, GPU arbiter, telemetry, config layering, adaptive barge-in, gain schedule. No hardware needed.

Expected: **30 passed**.


In [ ]:
!python3 -m pytest tests/unit/ tests/integration/ -q


## Stage 3 — Per-subsystem debug GUI

One window, one tab per subsystem. Walk them in order; green before you move on.

| Tab | Verify |
| --- | --- |
| **Audio** | Mic waveform + RMS move when you speak. DOA polar plot swings toward your voice. |
| **STT** | 5 s record + transcribe returns text in well under a second. Look for `CUDAExecutionProvider` in the startup log — that's the new shared ONNX factory active. |
| **LLM** | Loads Gemma-4 without OOM. Chat produces a reply. |
| **Tools** | FunctionGemma routes *"set a timer for 5 minutes"* to the timer tool. |
| **Memory** | Add / search / forget work for a test speaker. |
| **TTS** | Kokoro synth with RTF < 1.0. |
| **Vision** | YOLO pose detects face bbox; HSEmotion shows valence/arousal changing as your expression changes. |
| **VLM** | *(Optional — uses standalone Moondream; production uses Gemma multimodal instead.)* |
| **Face** | ESP32 renders all 11 preset expressions. |
| **Face tracking** | Head tracks your face smoothly. Kp/deadband sliders work. |
| **Motor control** | Manual 3-D head drag respects soft limits. |

This runs Qt — run in a terminal if the notebook cell hangs.


In [ ]:
!python3 -m tests.debug_gui

## Stage 4 — Combined CLI sanity check

Runs every subsystem's CLI sanity check sequentially and prints a summary. Good for catching regressions after any config change.


In [ ]:
!python3 -m tests.cli all

## Stage 5 — Full app, **simulated motors**

First end-to-end run. Motors are forced into the simulator bus so nothing physical moves — safe even if the motor serial cable isn't plugged in.

Before this cell, edit `config.yaml`:
```yaml
motor:
  enabled: true
  sim_only: true          # ← sim for this stage
```

Then run `main.py` in a **terminal** (Qt doesn't work in notebooks):

```bash
source companion_env/bin/activate
python3 main.py
```

What to verify:

1. **Startup log** shows:
   - `Readiness probe` all OK
   - `Conversation started (mode=continuous)`
   - `BehaviorEngine started`
   - `HealthMonitor started`
   - `Ready. Walk into view and say something…`
2. **Idle** — let it sit 30 s. No errors. `logs/traces_*.jsonl` is empty (no turns yet).
3. **Stand in view** — `BehaviorEngine` logs a face-appeared event (indirectly via trace stamps).
4. **Speak naturally** — ~800 ms after you stop, the reply starts. A JSONL line appears in `logs/traces_*.jsonl`.
5. **Mid-sentence pause** — pause for ~500 ms mid-sentence; robot should **not** interrupt. (VAD silence window + EOU wait absorb pauses up to ~1.1 s.)
6. **Interrupt mid-reply** — start talking while it's replying; it should stop within ~300 ms and listen.
7. **Walk out of view and speak** — robot should **not** reply. Engagement gate rejects because no face.
8. **Touchscreen tiles** — tap each one and confirm the expected behaviour:
   - Mute mic → log says `Mic muted: True`; robot ignores further speech; tap again to unmute.
   - Stop → cancels any in-flight reply.
   - Sleep → soft idle (returns to IDLE_WATCHING).
   - More → Volume (announces current %), Restart (exits cleanly), Back (returns to face).
9. **Ctrl-C** — clean shutdown, exits in under 5 s. Log shows `Shutting down…` and the `_ShutdownOrder` walks subsystems in reverse order.

If any of these misbehave, check `logs/stderr.log` and the `HealthDegraded` events logged at WARNING level.


## Stage 6 — Full app, **real motors** + telemetry

Flip `config.yaml`:
```yaml
motor:
  enabled: true
  sim_only: false         # ← real hardware
  home_on_startup: true
```

**Before running:** clear space around the head so it can home (drive to 0°, 0°) without hitting anything. If you see overshoot, stop and re-run calibration: `python -m companion.motor.cli calibrate --check`.

Run `main.py` in a terminal again:

```bash
python3 main.py
```

Repeat every check from Stage 5, plus:

- **Smooth tracking** — walk slowly left-to-right. Head follows without jerking.
- **Thinking freeze** — during `THINKING` the head should **hold still**. That's the `kp=0.0` profile in `companion/behavior/tracking.py`.
- **Face lost mid-reply** — step out of view mid-reply. Robot finishes the current sentence, then falls silent after ~8 s of continued absence (Coordinator watcher kicks in).

### Telemetry — the 800 ms target

After a few turns, quit and inspect the JSONL traces:


In [ ]:
import json, glob
from pathlib import Path

# Each turn writes one JSON line with phase timestamps.
# t_vad_end  → t_first_audio  is the end-to-end "end-of-speech to first audio" latency.

latencies_ms = []
for path in sorted(Path('logs').glob('traces_*.jsonl')):
    for line in open(path):
        tr = json.loads(line)
        if tr.get('t_vad_end') and tr.get('t_first_audio'):
            ms = (tr['t_first_audio'] - tr['t_vad_end']) * 1000.0
            latencies_ms.append(ms)
            print(f"{tr['turn_id']}: end→first_audio = {ms:6.0f} ms  route={tr.get('route'):6}  text={tr.get('transcript','')[:60]!r}")

if latencies_ms:
    latencies_ms.sort()
    p50 = latencies_ms[len(latencies_ms)//2]
    p95 = latencies_ms[int(len(latencies_ms) * 0.95)]
    print(f"\np50 = {p50:.0f} ms   p95 = {p95:.0f} ms   (target: p95 < 900 ms)")
else:
    print("No turn traces yet. Have a conversation first, then re-run this cell.")


### If p95 is above 900 ms

Common causes:

- **CUDA not actually active.** Check startup logs for `CUDAExecutionProvider available`. If not, `onnxruntime-gpu` may be missing or the CUDA wheel of `llama-cpp-python` isn't installed.
- **Short-opener instruction not landing.** The system prompt tells Gemma to keep the first sentence under 8 words — if it's being ignored, TTS has to wait for a longer first sentence. You can verify by eyeballing the first sentence in reply transcripts.
- **KV-cache not being reused.** `llama-cpp-python` version matters; `≥ 0.2.80` has reliable prefix caching. If upgrading isn't an option, the LLM prefill call during STT tail still buys back most of the latency.

### If engagement feels too strict (robot ignores you too often)

In `config.yaml`:
```yaml
conversation:
  engagement:
    require_face: false              # accept audio without needing a visible face
    min_speech_ms: 300               # lower = more sensitive
    doa_face_concordance_deg: 30     # looser DOA gate
vad:
  min_speech_duration_ms: 300        # keeps step with engagement.min_speech_ms
```

### If the robot keeps cutting you off

Bump the silence window + EOU wait:
```yaml
vad:
  silence_duration_ms: 700           # wait 700 ms of silence before speech_end
eou:
  extra_wait_ms: 900                 # extra grace when EOU thinks you're not done
```

Total pause tolerance becomes ~1.6 s — good for slow speakers.
